# Production Inference Frameworks for Large Language Models

When deploying LLMs in production, raw model weights and a simple `model.generate()` call are rarely sufficient. Production workloads demand **low latency**, **high throughput**, and **efficient memory usage** — especially when serving many concurrent users.

This notebook covers three leading frameworks purpose-built for production inference:

| Framework | Key Strength | Runs On |
|-----------|-------------|---------|
| **Text Generation Inference (TGI)** | Stable, predictable GPU serving with Flash Attention | GPU |
| **vLLM** | Smart memory management via Paged Attention | GPU |
| **llama.cpp** | Lightweight CPU inference with quantized models | CPU / laptop |

> **Scope:** This notebook focuses on *how these frameworks optimise inference at scale*, not on single-machine usage tutorials.

---

## 1. Text Generation Inference (TGI)

**TGI** is Hugging Face's production-grade library for serving LLMs. It is designed to be **stable and predictable** under real-world traffic.

### Key Design Choices

- **Fixed sequence lengths** — keeps memory usage consistent and avoidable allocation spikes.
- **Flash Attention 2** — accelerates the attention computation while drastically reducing memory overhead.
- **Continuous batching** — efficiently groups incoming requests to maximise GPU utilisation.

### Flash Attention — The Core Optimisation

Standard attention computes the full **Q·Kᵀ** matrix, writes it to GPU high-bandwidth memory (HBM), then reads it back for the softmax and multiplication with **V**. This constant shuttling between compute units and memory is the main bottleneck — the GPU spends more time *waiting on data transfers* than actually computing.

**Flash Attention** eliminates this bottleneck by:

1. **Tiling** — breaking Q, K, V into small blocks that fit in fast on-chip SRAM.
2. **Fused kernels** — computing softmax and the attention output *in one pass* without ever materialising the full Q·Kᵀ matrix in HBM.
3. **Online softmax** — an incremental algorithm that computes exact softmax across tiles without needing the full row.

```
┌──────────────────────────────────────────────────────────────────────┐
│                    STANDARD ATTENTION                                │
│                                                                      │
│   Q, K, V  ──▶  Compute Q·Kᵀ  ──▶  Write to HBM  ──▶  Read back   │
│                                      (full N×N)        from HBM     │
│                  ──▶  Softmax  ──▶  Write to HBM  ──▶  Read back    │
│                  ──▶  × V      ──▶  Output                          │
│                                                                      │
│   ⚠️  Multiple slow HBM round-trips — GPU idles waiting on memory    │
└──────────────────────────────────────────────────────────────────────┘

┌──────────────────────────────────────────────────────────────────────┐
│                    FLASH ATTENTION                                    │
│                                                                      │
│   Q, K, V  ──▶  Load small tiles into SRAM                          │
│             ──▶  Compute attention + softmax in SRAM (fused)         │
│             ──▶  Write final output to HBM                           │
│                                                                      │
│   ✅  One pass, no large intermediates — GPU stays busy computing    │
└──────────────────────────────────────────────────────────────────────┘
```

### Analogy: Cooking a Meal 🍳

| | Standard Attention | Flash Attention |
|---|---|---|
| **Step 1** | Chop all ingredients → store in fridge | Chop a small batch |
| **Step 2** | Cook everything → store in fridge | Cook it immediately in one pan |
| **Step 3** | Mix all together → store in fridge | Serve directly |
| **Result** | Kitchen cluttered, constant trips to fridge (slow) | Everything done in-place, minimal storage (fast) |

### Benefits of Flash Attention for Production

- **Higher throughput** — serve more users with the same hardware.
- **Longer context windows** — handle longer sequences without running out of VRAM.
- **Lower VRAM usage** — memory scales **O(N)** instead of **O(N²)**.
- **Faster time-to-first-token** — reduced latency for end users.

> **Takeaway:** Flash Attention is a *memory-IO optimisation* — it computes the exact same result as standard attention, just without the wasteful data movement.

---

## 2. vLLM — Paged Attention

**vLLM** is a high-throughput inference engine that tackles one of the biggest memory problems in LLM serving: **KV cache management**.

### The Problem

During autoregressive generation, the model stores a **Key-Value (KV) cache** — a record of past tokens' representations so they don't need to be recomputed at each step.

- The KV cache **grows with every new token** generated.
- With **multiple concurrent users**, memory consumption explodes.
- Traditional allocators reserve **contiguous blocks** of memory, leading to **fragmentation and waste**.

```
┌─────────────────────────────────────────────────────────┐
│              KV CACHE MEMORY PROBLEM                     │
│                                                          │
│  GPU Memory:                                             │
│  ┌────────┬──────┬────────┬──────┬────────┬──────────┐  │
│  │ User A │ FREE │ User B │ FREE │ User C │  WASTED  │  │
│  │ cache  │      │ cache  │      │ cache  │  (frag.) │  │
│  └────────┴──────┴────────┴──────┴────────┴──────────┘  │
│                                                          │
│  ⚠️  Contiguous allocation → gaps between users          │
│  ⚠️  Pre-allocated max length → most space wasted        │
└─────────────────────────────────────────────────────────┘
```

### The Solution: Paged Attention

Paged Attention (inspired by OS virtual memory) breaks the KV cache into **small, fixed-size pages** that can be stored **anywhere** in GPU memory — no contiguous allocation required.

**How it works:**

1. The KV cache for each request is split into **pages** (e.g., 16 tokens per page).
2. Pages are allocated **on demand** as the sequence grows.
3. A **page table** (lookup table) maps each request to its scattered pages.
4. Pages from finished requests are **instantly recycled**.

```
┌─────────────────────────────────────────────────────────┐
│              PAGED ATTENTION SOLUTION                     │
│                                                          │
│  GPU Memory (pages):                                     │
│  ┌────┬────┬────┬────┬────┬────┬────┬────┬────┬────┐    │
│  │ A₁ │ B₁ │ C₁ │ A₂ │ B₂ │ C₂ │ A₃ │ B₃ │free│free│    │
│  └────┴────┴────┴────┴────┴────┴────┴────┴────┴────┘    │
│                                                          │
│  Page Table:                                             │
│    User A → [A₁, A₂, A₃]                                │
│    User B → [B₁, B₂, B₃]                                │
│    User C → [C₁, C₂]                                    │
│                                                          │
│  ✅  No fragmentation — pages fill any free slot         │
│  ✅  Shared prefixes can reuse the same pages            │
└─────────────────────────────────────────────────────────┘
```

### Flash Attention vs. Paged Attention

These two optimisations are **complementary**, not competing:

| | Flash Attention | Paged Attention |
|---|---|---|
| **Optimises** | Attention *computation* speed | KV cache *memory management* |
| **Problem solved** | Slow HBM data transfers | Memory fragmentation & waste |
| **Mechanism** | Tiled, fused SRAM kernels | Virtual-memory-style page tables |
| **Used by** | TGI, vLLM, and others | vLLM (primary), TGI (adopted) |

> **Key insight:** Flash Attention makes each attention operation *faster*. Paged Attention lets you *fit more requests* in memory. Together, they enable high-throughput, low-latency serving.

---

## 3. llama.cpp — LLMs on Consumer Hardware

**llama.cpp** is a lightweight C/C++ inference engine that lets you run large language models **on a CPU** — even on a standard laptop — by aggressively reducing model size.

### Why It Matters

Full-precision LLMs (FP16/FP32) are designed for high-end GPUs and can require **tens of gigabytes** of VRAM. Most developers and end users don't have access to that hardware.

### How llama.cpp Solves This

| Technique | What It Does |
|-----------|-------------|
| **Quantisation (4-bit / 8-bit)** | Shrinks each weight from 16 or 32 bits down to 4–8 bits, reducing model size by **4–8×**. |
| **Pure C/C++ implementation** | No Python overhead, no CUDA dependency — runs on any CPU. |
| **GGUF format** | A custom model format optimised for fast memory-mapped loading. |
| **Optional GPU offload** | Can offload select layers to a GPU if one is available, for a speed boost. |

```
┌────────────────────────────────────────────────────────────┐
│               MODEL SIZE: QUANTISATION IMPACT               │
│                                                             │
│   LLaMA 7B (FP16)    ████████████████████████  ~13 GB      │
│   LLaMA 7B (8-bit)   ████████████             ~7 GB        │
│   LLaMA 7B (4-bit)   ██████                   ~3.5 GB      │
│                                                             │
│   ✅  4-bit models fit in laptop RAM with minimal           │
│      quality loss for most tasks                            │
└────────────────────────────────────────────────────────────┘
```

---

## Summary: Choosing the Right Framework

```
┌─────────────────────────────────────────────────────────────────┐
│                  DECISION GUIDE                                  │
│                                                                  │
│   Need production GPU serving         ──▶  TGI or vLLM          │
│   with stability & HF integration?         (TGI)                │
│                                                                  │
│   Need maximum throughput              ──▶  vLLM                 │
│   with many concurrent users?                                    │
│                                                                  │
│   Need to run on a laptop / CPU        ──▶  llama.cpp            │
│   without a GPU?                                                 │
│                                                                  │
│   Need both speed & memory             ──▶  vLLM (Flash +       │
│   efficiency on GPU?                        Paged Attention)     │
└─────────────────────────────────────────────────────────────────┘
```

| Feature | TGI | vLLM | llama.cpp |
|---------|-----|------|-----------|
| **Runtime** | GPU | GPU | CPU (+ optional GPU) |
| **Flash Attention** | ✅ | ✅ | ❌ |
| **Paged Attention** | ✅ (adopted) | ✅ (native) | ❌ |
| **Quantisation** | Limited | Limited | Core feature (4/8-bit) |
| **HF Hub integration** | Native | Good | Via GGUF conversion |
| **Best for** | Stable HF model serving | High-throughput serving | Edge / local inference |

---

### References & Further Reading

- [Flash Attention Paper (Dao et al., 2022)](https://arxiv.org/abs/2205.14135)
- [Paged Attention / vLLM Paper (Kwon et al., 2023)](https://arxiv.org/abs/2309.06180)
- [TGI Documentation](https://huggingface.co/docs/text-generation-inference)
- [llama.cpp GitHub](https://github.com/ggerganov/llama.cpp)

---

### Implementation

In [ ]:
# Clone the repository (skip if already cloned)
!git clone https://github.com/ggerganov/llama.cpp 2>/dev/null || echo "Already cloned"

# Build with CMake (Make is no longer supported)
!cd llama.cpp && cmake -B build && cmake --build build --config Release -j

# Download the SmolLM2-1.7B-Instruct-GGUF model (note: lowercase filename on HF Hub)
!cd llama.cpp && curl -L -o smollm2-1.7b-instruct.Q4_K_M.gguf \
    "https://huggingface.co/HuggingFaceTB/SmolLM2-1.7B-Instruct-GGUF/resolve/main/smollm2-1.7b-instruct-q4_k_m.gguf"

-- The C compiler identification is AppleClang 21.0.0.21000099
-- The CXX compiler identification is AppleClang 21.0.0.21000099
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /Library/Developer/CommandLineTools/usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /Library/Developer/CommandLineTools/usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.50.1 (Apple Git-155)")
-- The ASM compiler identification is AppleClang
-- Found assembler: /Library/Developer/CommandLineTools/usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster 

In [20]:
import subprocess, time

# Start llama.cpp server in the background
server_process = subprocess.Popen(
    [
        "./llama.cpp/build/bin/llama-server",
        "-m", "./llama.cpp/smollm2-1.7b-instruct.Q4_K_M.gguf",
        "--host", "0.0.0.0",
        "--port", "8080",
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

# Give the server a few seconds to start
time.sleep(5)
print(f"llama.cpp server started (PID: {server_process.pid})")

llama.cpp server started (PID: 674)


In [ ]:
from openai import OpenAI

# Initialize OpenAI client pointing to local llama.cpp server
client = OpenAI(
    base_url="http://localhost:8080/v1",
    api_key="sk-no-key-required",
)

# Chat completion
response = client.chat.completions.create(
    model="smollm2-1.7b-instruct",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Tell me a story"},
    ],
    max_tokens=100,
    temperature=0.7,
    top_p=0.95,
)
print(response.choices[0].message.content)

# Text completion
response = client.completions.create(
    model="smollm2-1.7b-instruct",
    prompt="Tell me a story",
    max_tokens=100,
    temperature=0.7,
)
print(response.choices[0].text)

HfHubHTTPError: Client error '404 Not Found' for url 'http://localhost:8080/v1'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404

{'message': 'File Not Found', 'type': 'not_found_error', 'code': 404}